In [ ]:
!pip install "labtasker[plugins]" pandas

In [1]:
import os
os.chdir("/path/to/starVLA")
print(os.getcwd())
from labtasker import ls_tasks

/path/to/starVLA


In [2]:
!labtasker task count -f "metadata.benchmark == 'RoboTwin'"

🟡 Pending: 0
🔵 Running: 0
🟢 Success: >= 100
🔴 Failed: 0
⚪ Cancelled: 0


In [3]:
# Filter by benchmark to avoid mixing with LIBERO tasks
# Add more conditions as needed:
# e.g. extra_filter = 'metadata.benchmark == "RoboTwin" and args.policy_name == "my_run_v1"'
# e.g. extra_filter = 'metadata.benchmark == "RoboTwin" and args.mode == "demo_clean"'
extra_filter = 'metadata.benchmark == "RoboTwin"'

tasks = ls_tasks(
    status="success",
    extra_filter=extra_filter if extra_filter else None,
    limit=1000,
).content
print(f"{len(tasks)} successful tasks")

100 successful tasks


In [4]:
tasks[0]

Task(unknown_fields={}, task_id='2741f37f-b292-492c-ad2c-dce16e2d4296', queue_id='b5b7ed91-b5ef-46bc-a244-33216576b5d9', status='success', task_name='Qwen3-VL-OFT-RoboTwin2-All_beat_block_hammer_demo_randomized', created_at=datetime.datetime(2026, 4, 28, 16, 59, 5, 313000), start_time=datetime.datetime(2026, 4, 28, 18, 25, 10, 150000), last_heartbeat=datetime.datetime(2026, 4, 28, 20, 3, 0, 761000), last_modified=datetime.datetime(2026, 4, 28, 20, 3, 5, 556000), heartbeat_timeout=30.0, task_timeout=None, max_retries=3, retries=0, priority=10, metadata={'benchmark': 'RoboTwin', 'task_name': 'beat_block_hammer', 'mode': 'demo_randomized', 'policy_name': 'Qwen3-VL-OFT-RoboTwin2-All'}, args={'task_name': 'beat_block_hammer', 'mode': 'demo_randomized', 'ckpt': '/path/to/starVLA/.cache/huggingface/hub/models--StarVLA--Qwen3-VL-OFT-RoboTwin2-All/snapshots/2d8a0487b501c15bc96569f4ec7c09c8aba76820/checkpoints/steps_140000_pytorch_model.pt', 'policy_name': 'Qwen3-VL-OFT-RoboTwin2-All'}, cmd=['ex

In [5]:
import pandas as pd

rows = []
for t in tasks:
    rows.append({
        "policy_name": t.args["policy_name"],
        "task_name":   t.summary["task_name"],
        "mode":        t.summary["mode"],
        "success_rate": t.summary["success_rate"],
    })

df = pd.DataFrame(rows)
df = df.set_index(["policy_name", "task_name", "mode"]).sort_index()
df

success_rate
policy_name                task_name          mode                         
Qwen3-VL-OFT-RoboTwin2-All adjust_bottle      demo_clean               1.00
                                              demo_randomized          0.99
                           beat_block_hammer  demo_clean               0.94
                                              demo_randomized          0.94
                           blocks_ranking_rgb demo_clean               0.99
...                                                                     ...
                           stack_bowls_two    demo_randomized          0.97
                           stamp_seal         demo_clean               0.84
                                              demo_randomized          0.92
                           turn_switch        demo_clean               0.69
                                              demo_randomized          0.61

[100 rows x 1 columns]

In [6]:
# Per-policy aggregate across all tasks and modes
summary_rows = []
for (policy_name, mode), grp in df.groupby(["policy_name", "mode"]):
    summary_rows.append({
        "policy_name": policy_name,
        "mode":        mode,
        "n_tasks":     len(grp),
        "avg_success_rate": grp["success_rate"].mean(),
    })

summary_df = pd.DataFrame(summary_rows).set_index(["policy_name", "mode"])
summary_df.sort_values("avg_success_rate", ascending=False)

n_tasks  avg_success_rate
policy_name                mode                                      
Qwen3-VL-OFT-RoboTwin2-All demo_randomized       50            0.8884
                           demo_clean            50            0.8848